# Day 14 · Pandas 进阶

> **前置**: Day13 已掌握 DataFrame 创建/查看/loc/iloc
> **关键问题**: 如何从 10 万行中筛选出"年龄>30 且 城市=北京"? 如何按城市分组计算平均工资? 多表如何像 SQL 一样 join?

**引入:从"看数"到"问数"**

抽问: `df.loc[1:3]` 和 `df.iloc[1:3]` 取的行数一样吗?(不一样, loc 含右端取 3 行, iloc 不含右端取 2 行)上一节学会了"看"数据到某行某列 
—— 这一节学会"问"数据,让数据回答你的业务问题。

本节核心方法一览: **布尔过滤** → 挑出满足条件的行; **排序** → 让数据有序排列; **groupby** → 分组后做聚合计算; **agg** → 多指标聚合 + 
命名输出列; **merge** → 像 SQL 一样 join 多张表; **concat** → 拼接多张表; **pivot_table** → 透视表(交叉分析); 
**fillna / dropna** → 处理缺失值。


**1. 布尔索引过滤**

**布尔索引过滤** 是用条件表达式生成 True/False 掩码, 再据此选行:
- 单条件: `df[df["成绩"] >= 80]`
- 多条件: `&` 表示 "且", `|` 表示 "或", **每个条件必须用括号**
- **query()**: 写字符串条件, 更简洁, 用 `@变量` 引用外部变量

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "班级": ["A", "A", "B", "B", "A"],
    "成绩": [85, 92, 78, 88, 65],
    "性别": ["男", "女", "男", "女", "男"]
})
print("原始数据:"); print(df)

# --- 单条件: 成绩 >= 80 ---
df_a = df[df["成绩"] >= 80]
print("\n成绩>=80:"); print(df_a)

# --- 多条件: 且(&) 每个条件必须加括号 ---
# 🔴 不加括号会报错: (df["成绩"] >= 80) & (df["班级"] == "A")
df_b = df[(df["成绩"] >= 80) & (df["班级"] == "A")]
print("\n成绩>=80 且 班级=A:"); print(df_b)

# --- 多条件: 或(|) ---
df_c = df[(df["成绩"] >= 90) | (df["班级"] == "B")]
print("\n成绩>=90 或 班级=B:"); print(df_c)

# --- query(): 字符串表达式,更简洁 ---
df_q1 = df.query("成绩 >= 80 and 班级 == 'A'")
print("\nquery 等价写法:"); print(df_q1)

# --- query() 用 @引用外部变量 ---
min_score = 80
df_q2 = df.query("成绩 > @min_score")
print("\nquery + @变量(min_score=80):"); print(df_q2)
"""
小结:
- 过滤本质: df[布尔Series]
- 多条件: (条件1) & (条件2) 括号必加
- query("...") 写法更清爽,@变量 引用外部
"""

**✏️ 练一练 ⭐**

某学生成绩表如下,请筛选出:(1) 成绩在 80 到 90 之间(含端点);(2) 班级为 B 或性别为 女。分别用 `&`/`|` 写法实现。

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "班级": ["A", "A", "B", "B", "A"],
    "成绩": [85, 92, 78, 88, 65],
    "性别": ["男", "女", "男", "女", "男"]
})

# ======================
# 学员代码区
# ======================
# 练习1: 筛选 80 <= 成绩 <= 90
df_1 = None  # 请在此填写

# 练习2: 筛选 班级==B 或 性别==女
df_2 = None  # 请在此填写

print("成绩在80~90之间:"); print(df_1)
print("\n班级B或性别女:"); print(df_2)
pass

In [ ]:
# ======================
# 参考答案
# ======================
# 练习1: (成绩>=80) & (成绩<=90)
df_1 = df[(df["成绩"] >= 80) & (df["成绩"] <= 90)]

# 练习2: (班级==B) | (性别==女)
df_2 = df[(df["班级"] == "B") | (df["性别"] == "女")]

print("成绩在80~90之间:"); print(df_1)
print("\n班级B或性别女:"); print(df_2)

**2. 排序**

**排序** 用 `sort_values` 按某列值重排:
- `ascending=True` 升序(默认), `False` 降序
- 多列排序: 传列表, `ascending` 也传列表
- 排序不改变原 DataFrame, 返回新 DataFrame(需赋值)

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "班级": ["A", "A", "B", "B", "A"],
    "成绩": [85, 92, 78, 88, 65],
    "性别": ["男", "女", "男", "女", "男"]
})
print("原始数据:"); print(df)

# --- 排序: 单字段降序 ---
df_s1 = df.sort_values("成绩", ascending=False)
print("\n按成绩降序:"); print(df_s1)

# --- 排序: 多字段不同升降序 ---
# 先按班级升序,班级相同时按成绩降序
df_s2 = df.sort_values(["班级", "成绩"],
                        ascending=[True, False])
print("\n班级升序+成绩降序:"); print(df_s2)
"""
小结:
- sort_values(列, ascending= ...) 默认升序
- 多列时传列表,ascending 也传等长列表
- 返回新 DataFrame,原 df 不变(需要赋值接收)
"""

**✏️ 练一练 ⭐**

对上述学生表,先按班级升序,同一班级内按成绩降序排列,输出前 3 名同学的姓名和成绩。

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "班级": ["A", "A", "B", "B", "A"],
    "成绩": [85, 92, 78, 88, 65],
    "性别": ["男", "女", "男", "女", "男"]
})

# ======================
# 学员代码区
# ======================
# 请排序后取前3名的姓名和成绩
result = None  # 请在此填写
print(result)
pass

In [ ]:
# ======================
# 参考答案
# ======================
result = (df.sort_values(["班级", "成绩"],
                          ascending=[True, False])
            .head(3)[["姓名", "成绩"]])
print(result)

**3. 分组聚合 groupby**

**核心思想**: split → apply → combine(分组 → 对每组做运算 → 合并结果)
- `df.groupby("列")["值列"].mean()` 分组后取某列做聚合
- 多列分组: 传列表 `groupby(["班级","性别"])`
- **建议先选列再聚合**: `groupby("班级")["成绩"].mean()` 比 `groupby("班级").mean()` 更精准,避免对非数值列误操作

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "班级": ["A", "A", "A", "B", "B", "B"],
    "性别": ["男", "女", "男", "女", "男", "女"],
    "成绩": [85, 92, 78, 88, 76, 95]
})
print("原始数据:"); print(df)

# --- 基本: 按班级分组取平均成绩 ---
# 🔴 推荐先选列再聚合,避免对非数值列误操作
g1 = df.groupby("班级")["成绩"].mean()
print("\n各班平均成绩:"); print(g1)

# --- 多列分组: 班级+性别 ---
g2 = df.groupby(["班级", "性别"])["成绩"].mean()
print("\n各班男女平均:"); print(g2)
"""
小结:
- groupby("列") 返回 DataFrameGroupBy 对象,本身不是结果
- 接聚合方法: .mean() / .sum() / .count() / .max() / .min()
- 先选列再聚合更精准: ["列"].mean()
"""

**4. agg 多聚合函数**

**agg()** 可以同时定制多种聚合方式:
- 列表形式: `agg(["mean","max","min","count"])` — 自动生成列名
- **命名聚合语法**: `agg(新名=("列","函数"))` — 自定义输出列名,推荐在报告中使用

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "班级": ["A", "A", "A", "B", "B", "B"],
    "性别": ["男", "女", "男", "女", "男", "女"],
    "成绩": [85, 92, 78, 88, 76, 95]
})

# --- agg + 列表: 一次算多个指标 ---
g3 = df.groupby("班级")["成绩"].agg(
    ["mean", "max", "min", "count"]
)
print("多指标聚合(列表):"); print(g3)

# --- 命名聚合语法: 输出列名自定义 ---
# 📌 格式: 新列名=("原列名", "聚合函数名")
g4 = df.groupby("班级").agg(
    平均=("成绩", "mean"),
    最高=("成绩", "max"),
    人数=("成绩", "count")
)
print("\n命名聚合(中文列名):"); print(g4)
"""
小结:
- agg(["mean","max"]) 列表快速看多指标
- agg(新名=("列","函数")) 命名聚合输出更清晰
- 命名聚合是官方推荐写法,适合报表输出
"""

**✏️ 练一练 ⭐⭐**

某销售数据含"城市"和"销售额"两列,请按城市分组,用命名聚合同时计算:平均销售额、最高销售额、订单数。输出列名分别为: 平均、最高、订单数。

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "城市": ["北京", "北京", "上海", "上海", "广州"],
    "销售额": [120, 150, 80, 200, 90]
})

# ======================
# 学员代码区
# ======================
# 请用命名聚合实现
result = None  # 请在此填写
print(result)
pass

In [ ]:
# ======================
# 参考答案
# ======================
result = df.groupby("城市").agg(
    平均=("销售额", "mean"),
    最高=("销售额", "max"),
    订单数=("销售额", "count")
)
print(result)

**5. merge 合并**

**merge** 像 SQL 的 join, 根据关联字段把左右两表拼起来:
- `how="inner"`: 只留两边都有的键(默认)
- `how="left"`: 以左表为主, 右表匹配不到填 NaN
- `how="right"`: 以右表为主
- `how="outer"`: 两边键的并集
- **merge 可能改变行数** — 合并后务必检查 `.shape`

In [ ]:
import pandas as pd

# 两张表通过 学号 关联
df_stu = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S4"],
    "姓名": ["张三", "李四", "王五", "赵六"]
})
df_score = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S5"],
    "数学": [90, 85, 78, 88]
})
print("学生表:"); print(df_stu)
print("\n成绩表:"); print(df_score)

# --- inner join (默认): 只留两边都有的学号 ---
m_inner = pd.merge(df_stu, df_score,
                   on="学号", how="inner")
print("\ninner join:"); print(m_inner)
print("shape:", m_inner.shape)  # (3, 3) S5 不在左表被丢弃

# --- left join: 以左表为主,右表不到填 NaN ---
m_left = pd.merge(df_stu, df_score,
                  on="学号", how="left")
print("\nleft join:"); print(m_left)
print("shape:", m_left.shape)  # (4, 3) 左表 S4 保留,数学 NaN

# --- right join: 以右表为主 ---
m_right = pd.merge(df_stu, df_score,
                   on="学号", how="right")
print("\nright join:"); print(m_right)

# --- outer join: 两边键的并集 ---
m_outer = pd.merge(df_stu, df_score,
                   on="学号", how="outer")
print("\nouter join:"); print(m_outer)
"""
小结:
- on= 指定关联键(两表列名相同时)
- how= 控制合并策略: inner/left/right/outer
- merge 可能增减行数,务必 .shape 确认
"""

**✏️ 练一练 ⭐⭐**

"学生表"含 学号/姓名,"班级表"含 学号/班级。请用 left join 合并,观察合并后 shape 是多少,并找出班级为空的学生。

In [ ]:
import pandas as pd

df_stu = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S4", "S5"],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"]
})
df_cls = pd.DataFrame({
    "学号": ["S1", "S2", "S3"],
    "班级": ["A", "A", "B"]
})

# ======================
# 学员代码区
# ======================
# 1. left join 合并
merged = None  # 请填写
print("合并结果:"); print(merged)
print("shape:", merged.shape)

# 2. 找出班级为空的学生
missing = None  # 请填写
print("\n班级为空:"); print(missing)
pass

In [ ]:
# ======================
# 参考答案
# ======================
merged = pd.merge(df_stu, df_cls,
                  on="学号", how="left")
print("合并结果:"); print(merged)
print("shape:", merged.shape)  # (5, 3)

# 班级为空的行
missing = merged[merged["班级"].isnull()]
print("\n班级为空:"); print(missing[["学号", "姓名"]])

**6. concat 拼接**

**concat** 沿轴拼接, 不依赖键:
- `axis=0`(默认): 纵向堆叠行,建议加 `ignore_index=True` 重置索引
- `axis=1`: 横向拼列
- 拼接不检查键, 可能出现重复索引或 NaN

In [ ]:
import pandas as pd

df1 = pd.DataFrame({"A": [1, 2], "B": [3, 4]})
df2 = pd.DataFrame({"A": [7, 8], "B": [9, 10]})
print("df1:"); print(df1)
print("\ndf2:"); print(df2)

# --- concat 纵向堆叠: ignore_index 重置索引 ---
c_v = pd.concat([df1, df2], ignore_index=True)
print("\nconcat axis=0 (ignore_index):"); print(c_v)

# --- concat 横向拼列: axis=1 ---
c_h = pd.concat([df1, df2], axis=1)
print("\nconcat axis=1:"); print(c_h)
print("列名:", c_h.columns.tolist())  # 列名可能重复

# --- 拼接不同列时,缺失值自动填 NaN ---
df3 = pd.DataFrame({"A": [1, 2], "C": [5, 6]})
c_m = pd.concat([df1, df3], ignore_index=True)
print("\n拼接不同列:"); print(c_m)  # B/C 都有 NaN
"""
小结:
- concat axis=0 纵向堆叠,ignore_index=True 重置索引
- concat axis=1 横向拼列
- 列不一致时自动补 NaN(和 merge outer 类似)
"""

**7. pivot_table 透视表**

透视表把长表变宽交叉表:
- `index`: 行分组键
- `columns`: 列分组键
- `values`: 要聚合的值列
- `aggfunc`: 聚合函数, 如 "sum" / "mean"
- `margins=True`: 增加行列合计

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "日期": ["周一", "周一", "周二", "周二", "周三", "周三"],
    "城市": ["北京", "上海", "北京", "上海", "北京", "上海"],
    "销量": [120, 85, 96, 110, 75, 60]
})
print("原始数据:"); print(df)

# --- 透视表: 日期×城市 看销量 ---
pt = df.pivot_table(
    index="日期",
    columns="城市",
    values="销量",
    aggfunc="sum"
)
print("\n透视表(无合计):"); print(pt)

# --- 透视表 + 行列合计 ---
pt_m = df.pivot_table(
    index="日期",
    columns="城市",
    values="销量",
    aggfunc="sum",
    margins=True
)
print("\n透视表(含合计):"); print(pt_m)
"""
小结:
- pivot_table 把"长表"变"宽交叉表"
- index/columns 定义行列,values 定义填充值
- margins=True 加行列合计,方便汇总
"""

**✏️ 练一练 ⭐⭐**

上述销售数据,请用 pivot_table 得到"城市×日期"的平均销量交叉表(不带合计)。即 index="城市", columns="日期", aggfunc="mean"。

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "日期": ["周一", "周一", "周二", "周二", "周三", "周三"],
    "城市": ["北京", "上海", "北京", "上海", "北京", "上海"],
    "销量": [120, 85, 96, 110, 75, 60]
})

# ======================
# 学员代码区
# ======================
pt = None  # 请在此填写
print(pt)
pass

In [ ]:
# ======================
# 参考答案
# ======================
pt = df.pivot_table(
    index="城市",
    columns="日期",
    values="销量",
    aggfunc="mean"
)
print(pt)

**8. 缺失值处理**

实际数据经常缺值, 三步走:
- **发现**: `df.isnull()` 返回布尔矩阵, `df.isnull().sum()` 按列统计缺失数目
- **删除**: `df.dropna()` 丢掉含 NaN 行, 可用 `subset` 限定检查列, `thresh` 要求至少 N 个非 NaN
- **填充**: `df.fillna(常数)` 填固定值, 传字典不同列填不同值, `method="ffill"` 用上一行前向填充
- ⚠️ **fillna 默认返回新 DataFrame**, 需赋值接收或用 `inplace=True`

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "成绩": [85, np.nan, 78, np.nan],
    "班级": ["A", "A", "B", "B"]
})
print("原始数据:"); print(df)

# --- 发现缺失: isnull 布尔矩阵 ---
print("\nisnull():"); print(df.isnull())

# --- 每列缺失计数(必看第一步) ---
print("\n每列缺失数:")
print(df.isnull().sum())

# --- dropna: 删掉任何含 NaN 的行 ---
d1 = df.dropna()
print("\ndropna():"); print(d1)

# --- dropna + subset: 只看成绩列 ---
d2 = df.dropna(subset=["成绩"])
print("\ndropna(subset=['成绩']):"); print(d2)

# --- dropna + thresh: 至少2个非NaN才保留 ---
df2 = pd.DataFrame({
    "A": [1, np.nan, 3],
    "B": [np.nan, 2, 3],
    "C": [1, 2, 3]
})
d3 = df2.dropna(thresh=2)
print("\ndropna(thresh=2):"); print(d3)

# --- fillna: 填常数 0 ---
f1 = df.fillna(0)
print("\nfillna(0):"); print(f1)

# --- fillna: 字典按列填充 ---
# 成绩列填该列均值
f2 = df.fillna({"成绩": df["成绩"].mean()})
print("\nfillna(dict 列均值):"); print(f2)

# --- fillna: 前向填充 ffill ---
df3 = pd.DataFrame({"A": [1, np.nan, np.nan, 4]})
f3 = df3.fillna(method="ffill")
print("\nfillna(ffill):"); print(f3)
"""
小结:
- isnull().sum() 快速定位哪些列有缺失
- dropna(subset/thresh) 精细控制删行条件
- fillna(常数/字典/ffill) 多种填充策略
- fillna 默认返回新 df,需赋值: df = df.fillna(...)
"""

**✏️ 练一练 ⭐⭐**

给定含缺失值的 DataFrame(如下),请完成:
1. 统计每列缺失值数量;
2. 将"成绩"列的缺失值填充为该列均值;
3. 输出处理后每列缺失数,确认"成绩"列已无缺失。

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "成绩": [85, np.nan, 78, np.nan, 92],
    "班级": ["A", "A", "B", "B", "A"]
})

# ======================
# 学员代码区
# ======================
# 1. 统计每列缺失值数量
missing_before = None  # 请填写
print("处理前缺失数:"); print(missing_before)

# 2. 将成绩列缺失值填充为列均值
df_filled = None  # 请填写

# 3. 确认处理后缺失情况
missing_after = None  # 请填写
print("\n处理后缺失数:"); print(missing_after)
pass

In [ ]:
# ======================
# 参考答案
# ======================
# 1. 统计缺失
missing_before = df.isnull().sum()
print("处理前缺失数:"); print(missing_before)

# 2. 填充成绩列均值(用字典精准填充)
df_filled = df.fillna({"成绩": df["成绩"].mean()})

# 3. 确认
missing_after = df_filled.isnull().sum()
print("\n处理后缺失数:"); print(missing_after)
print("\n填充后数据:"); print(df_filled)

**📝 本节试题集**

当堂练:`course/lesson14/in_class/practice01-06.py`(6 道)
课后作业:`course/lesson14/homework/task01-03.py`(3 道)

**今日小结**

本节围绕 Pandas 数据处理的 8 个核心操作 —— **布尔过滤** → **排序** → **groupby** → **agg** → **merge** → 
**concat** → **pivot_table** → **缺失值处理**,覆盖从"问数"到"清洗数"的完整流程。

**核心操作速查表**

| 操作 | 写法 | 要点 |
| --- | --- | --- |
| **布尔过滤** | `df[条件]` | `&` 且、`|` 或,每个条件加括号 |
| **query()** | `df.query("成绩>80")` | `@变量` 引用外部量 |
| **sort_values()** | `df.sort_values("列")` | `ascending` 控制升降,可传列表 |
| **groupby()** | `df.groupby("列")["值"].mean()` | split→apply→combine |
| **agg()** | `.agg(["mean","max"])` | 命名聚合 `新名=("列","函数")` |
| **merge()** | `pd.merge(L,R,on="键",how=...)` | how=inner/left/right/outer |
| **concat()** | `pd.concat([df1,df2])` | axis=0/1,建议 ignore_index=True |
| **pivot_table()** | `df.pivot_table(index,columns,values)` | aggfunc 指定聚合,margins 加合计 |
| **isnull().sum()** | 缺失计数 | 必看第一步 |
| **dropna()** | `df.dropna(subset,thresh)` | 精细控制删行条件 |
| **fillna()** | `df.fillna(值/字典/ffill)` | 默认返回新 df,需赋值 |

**常见报错与易错点**

- `df[条件1 & 条件2]` → `&` 优先级歧义,必须写 `df[(条件1) & (条件2)]`
- merge 后行数变了却没检查 `.shape` → 可能导致后续分析偏差
- `groupby("列").mean()` 对含非数值列的 df 会报错 → 必须先 `["值列"]` 选出来
- `df.fillna(0)` 没赋值,原 df 并未改变 → 要写 `df = df.fillna(0)`
- `dropna(how="all")` 只删全为 NaN 的行 → 默认 `how="any"`,有一个 NaN 就删

**课后综合练习**

- ⭐ 对自定义 DataFrame 用 `(df["A"]>1) & (df["B"]<2)` 做双条件过滤, 打印结果。
- ⭐⭐ 对成绩表做 `groupby("班级")`, 用命名聚合计算平均分、最高分、人数, 输出列名为中文。
- ⭐⭐⭐ 两张订单表含缺失值: 先 merge 左连接, 再用 fillna 把金额缺失填列均值, 最后 pivot_table 透视 日期×商品 看总销售额。